Дані

In [1]:
import os, re, gc, glob, json, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from xgboost import XGBClassifier
from scipy import stats
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import QuantileTransformer

BUDGET_MIN = 40                      # бюджет пошуку на кожну модель, хвилин
os.makedirs("results", exist_ok=True)
dev = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7
del tr

N = len(df)
TRAIN_W, VAL_W, TEST_W = (0, 17), (18, 21), (22, 26)
inw = lambda r: ((df.week >= r[0]) & (df.week <= r[1])).values
m_tr, m_va, m_te = inw(TRAIN_W), inw(VAL_W), inw(TEST_W)
assert (m_tr | m_va | m_te).all() and not (m_tr & m_va).any() and not (m_va & m_te).any()
assert (m_tr.sum(), m_va.sum(), m_te.sum()) == (431125, 77090, 82325)

ts, tid = df.TransactionDT.values, df.TransactionID.values
order = np.lexsort((tid, ts)); rank = np.empty(N, dtype=np.int64); rank[order] = np.arange(N)
y = df.isFraud.values
assert y.sum() == 20663
split_of = np.where(m_tr, 0, np.where(m_va, 1, 2))

X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT", "day", "week"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
df["D1n"] = (df["D1"] - df["TransactionDT"] / 86400).round()      # після X: у таблицю не потрапляє
spw = float((y[m_tr] == 0).sum() / (y[m_tr] == 1).sum())

def metrics(y_true, p):
    k = int(y_true.sum()); o = np.argsort(-p)
    top1 = o[: max(1, int(0.01 * len(y_true)))]
    return {"pr_auc": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p),
            "recall@k": y_true[o[:k]].sum() / k, "precision@1%": y_true[top1].mean()}

major = int(xgb.__version__.split(".")[0])
DEV = dict(tree_method="hist", device="cuda") if major >= 2 else dict(tree_method="gpu_hist")
print("готово:", X.shape)

GPU: Tesla T4
готово: (590540, 434)


Граф u2, u3, u5, мітки сусідів, входи моделей

In [2]:
def build_edges(cols, K, W_days):
    valid = df[cols].notna().all(axis=1).values
    idx = np.flatnonzero(valid); sub = df.loc[valid, cols]
    S, D = [], []
    for pos in sub.groupby(cols, sort=False).indices.values():
        if len(pos) < 2: continue
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt = ts[rows]; n = len(rows)
        lo = np.maximum(np.arange(n) - K, np.searchsorted(tt, tt - W_days * 86400, side="left"))
        cnt = np.arange(n) - lo; tot = int(cnt.sum())
        if tot == 0: continue
        dst_l = np.repeat(np.arange(n), cnt)
        start = np.repeat(np.cumsum(cnt) - cnt, cnt)
        src_l = np.arange(tot) - start + np.repeat(lo, cnt)
        S.append(rows[src_l].astype(np.int32)); D.append(rows[dst_l].astype(np.int32))
    return np.concatenate(S), np.concatenate(D)

REL = {"u2": (["card1", "addr1"], 50, 30),
       "u3": (["card1", "addr1", "P_emaildomain"], 50, 30),
       "u5": (["card1", "addr1", "P_emaildomain", "D1n"], 50, 30)}
R_NAMES = list(REL); KMAX = {k: v[1] for k, v in REL.items()}
edges = {k: build_edges(*v) for k, v in REL.items()}
for k, (s, d) in edges.items():
    assert (rank[s] < rank[d]).all() and (s != d).all()
    assert (split_of[s[split_of[d] == 0]] == 0).all()
    print(k, f"ребер {len(s):,}")

NBR, DEG = [], []
for rn in R_NAMES:
    s, d = edges[rn]
    o = np.argsort(d, kind="stable"); ds, ss = d[o], s[o]
    cnt = np.bincount(d, minlength=N); start = np.cumsum(cnt) - cnt
    pos = np.arange(len(ds)) - start[ds]
    assert pos.max() < KMAX[rn]
    tab = np.full((N, KMAX[rn]), -1, dtype=np.int32); tab[ds, pos] = ss
    NBR.append(torch.from_numpy(tab).to(dev))
    DEG.append(torch.from_numpy(cnt.astype(np.int64)).to(dev))

DL, W_LAB = 7, 30
def label_feats(cols, W_days, DLd):
    valid = df[cols].notna().all(axis=1).values; idx = np.flatnonzero(valid)
    nk = np.full(N, np.nan, "float32"); fk = np.full(N, np.nan, "float32")
    for pos in df.loc[valid, cols].groupby(cols, sort=False).indices.values():
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt, ff = ts[rows], y[rows]; cs = np.concatenate([[0], np.cumsum(ff)])
        a = np.searchsorted(tt, tt, side="left")
        hi = np.minimum(np.searchsorted(tt, tt - DLd * 86400, side="right"), a)
        lo = np.minimum(np.searchsorted(tt, tt - W_days * 86400, side="left"), hi)
        nk[rows] = hi - lo; fk[rows] = cs[hi] - cs[lo]
    return nk, fk

LF = {}
for rel, (cols, _, _) in REL.items():
    nk, fk = label_feats(cols, W_LAB, DL)
    LF[f"{rel}_nk"], LF[f"{rel}_fk"] = nk, fk
    LF[f"{rel}_rate"] = (fk / np.where(nk > 0, nk, np.nan)).astype("float32")
LF = pd.DataFrame(LF)

rng = np.random.default_rng(2); bad = 0
for rel, (cols, _, _) in REL.items():
    cand = np.flatnonzero(df[cols].notna().all(axis=1).values)
    for i in rng.choice(cand, 60, replace=False):
        same = np.ones(N, bool)
        for c in cols: same &= (df[c].values == df[c].values[i])
        kn = same & (ts <= ts[i] - DL * 86400) & (ts >= ts[i] - W_LAB * 86400)
        if (int(kn.sum()) != int(LF[f"{rel}_nk"].values[i]) or
                int(y[kn].sum()) != int(LF[f"{rel}_fk"].values[i])): bad += 1
print("розбіжностей з перерахунком «в лоб»:", bad); assert bad == 0

# ---- входи: для дерев (з NaN) і для нейромереж (нормалізовані) ----
Xn = X.values.astype("float32")
grp = lambda p: [i for i, c in enumerate(X.columns) if re.fullmatch(p, c)]
nanf = np.stack([np.isnan(Xn[:, g]).mean(1) for g in
                 (grp(r"V\d+"), grp(r"D\d+"), grp(r"id_\d+"), grp(r"M\d+"))], 1).astype("float32")
qt = QuantileTransformer(n_quantiles=200, output_distribution="normal",
                         subsample=200_000, random_state=0).fit(Xn[m_tr])
Xt = np.hstack([np.nan_to_num(qt.transform(Xn), nan=0.0).clip(-5, 5).astype("float32"), nanf])
cols_ = []
for r in R_NAMES:
    g = lambda k, r=r: np.nan_to_num(LF[f"{r}_{k}"].values)
    cols_ += [np.log1p(g("nk")), np.log1p(g("fk")), g("rate"), (g("nk") > 0).astype("float32")]
M = np.stack(cols_, 1).astype("float32")
M = (M - M[m_tr].mean(0)) / (M[m_tr].std(0) + 1e-6)
XG = torch.from_numpy(np.hstack([Xt, M])).half().to(dev)
YT = torch.as_tensor(y, device=dev).float()
XB_np = pd.concat([X, LF], axis=1).values.astype("float32")
del Xn; gc.collect()

tr_i, va_i, te_i = np.flatnonzero(m_tr), np.flatnonzero(m_va), np.flatnonzero(m_te)
Xtr_b, Xva_b, Xte_b = XB_np[tr_i], XB_np[va_i], XB_np[te_i]
ytr_b, yva_b, yte_b = y[tr_i], y[va_i], y[te_i]
print("входи готові: дерева", XB_np.shape, "| нейромережі", tuple(XG.shape))

u2 ребер 12,260,904
u3 ребер 5,295,877
u5 ребер 478,008
розбіжностей з перерахунком «в лоб»: 0
входи готові: дерева (590540, 443) | нейромережі (590540, 450)


Моделі й тест на витік

In [3]:
def sample(nodes, r, s, gen=None):
    deg = DEG[r][nodes]
    u = torch.rand(nodes.shape[0], s, device=dev, generator=gen)
    idx = (u * deg.unsqueeze(1)).long().clamp_(max=KMAX[R_NAMES[r]] - 1)
    nb = NBR[r][nodes.unsqueeze(1), idx].long()
    valid = (deg > 0).unsqueeze(1).expand(-1, s) & (nb >= 0)
    return nb.clamp(min=0), valid

def hops(tgt, s1, s2, gen=None):
    B, R = tgt.shape[0], len(R_NAMES)
    n1, v1 = zip(*[sample(tgt, r, s1, gen) for r in range(R)])
    N1, M1 = torch.stack(n1, 1), torch.stack(v1, 1)
    n2, v2 = zip(*[sample(N1.reshape(-1), r, s2, gen) for r in range(R)])
    N2 = torch.stack(n2, 1).view(B, R, s1, R, s2)
    M2 = torch.stack(v2, 1).view(B, R, s1, R, s2) & M1[:, :, :, None, None]
    return N1, M1, N2, M2

class RelSAGE(nn.Module):
    def __init__(self, F_in, H=128, R=3, drop=0.2, hops_n=2):
        super().__init__()
        self.R, self.hops_n = R, hops_n
        self.enc = nn.Sequential(nn.Linear(F_in, H), nn.ReLU(), nn.Dropout(drop))
        self.W1s, self.W2s = nn.Linear(H, H), nn.Linear(H, H)
        self.W1n = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(R)])
        self.W2n = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(R)])
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(H, 1))

    @staticmethod
    def mmean(e, m, dim):
        m = m.unsqueeze(-1).to(e.dtype)
        return (e * m).sum(dim) / m.sum(dim).clamp(min=1)

    def forward(self, XG, tgt, s1, s2, gen=None):
        B, R = tgt.shape[0], self.R
        enc = lambda ids: self.enc(XG[ids].float())
        if self.hops_n == 1:
            n1, v1 = zip(*[sample(tgt, r, s1, gen) for r in range(R)])
            N1, M1 = torch.stack(n1, 1), torch.stack(v1, 1)
            e_t = enc(tgt); e1 = enc(N1.reshape(-1)).view(B, R, s1, -1)
            a1 = self.mmean(e1, M1, 2)
            h = F.relu(self.W1s(e_t) + sum(self.W1n[j](a1[:, j]) for j in range(R)))
            return self.out(h).squeeze(-1)
        N1, M1, N2, M2 = hops(tgt, s1, s2, gen)
        e_t = enc(tgt)
        e1 = enc(N1.reshape(-1)).view(B, R, s1, -1)
        e2 = enc(N2.reshape(-1)).view(B, R, s1, R, s2, -1)
        a2 = self.mmean(e2, M2, 4)
        h1_u = F.relu(self.W1s(e1) + sum(self.W1n[j](a2[:, :, :, j]) for j in range(R)))
        a1 = self.mmean(e1, M1, 2)
        h1_t = F.relu(self.W1s(e_t) + sum(self.W1n[j](a1[:, j]) for j in range(R)))
        ah = self.mmean(h1_u, M1, 2)
        h2 = F.relu(self.W2s(h1_t) + sum(self.W2n[j](ah[:, j]) for j in range(R)))
        return self.out(h2).squeeze(-1)

class MLP(nn.Module):
    def __init__(self, F_in, hidden=128, layers=2, drop=0.2):
        super().__init__()
        mods, d = [], F_in
        for _ in range(layers):
            mods += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(drop)]; d = hidden
        self.net = nn.Sequential(*mods, nn.Linear(d, 1))
    def forward(self, XG, tgt, s1=None, s2=None, gen=None):
        return self.net(XG[tgt].float()).squeeze(-1)

# ---- тест на витік на реальних вибірках сусідів ----
sp_t = torch.as_tensor(split_of, device=dev); t_t = torch.as_tensor(ts, device=dev)
rg = np.random.default_rng(0)
for sp_, cap in [(0, 0), (1, 1)]:
    ids = torch.as_tensor(rg.choice(np.flatnonzero(split_of == sp_), 4000, replace=False), device=dev)
    N1, M1, N2, M2 = hops(ids, 10, 5)
    assert (sp_t[N1[M1]] <= cap).all() and (sp_t[N2[M2]] <= cap).all(), "сусіди з майбутньої частини"
    assert (t_t[N1] <= t_t[ids][:, None, None])[M1].all(), "сусід пізніший за вузол (крок 1)"
    assert (t_t[N2] <= t_t[N1][:, :, :, None, None])[M2].all(), "сусід пізніший за вузол (крок 2)"
print("тест на витік пройдено")

тест на витік пройдено


Тренування, простори пошуку, механізм пошуку

In [4]:
@torch.no_grad()
def predict_avg(model, idx_np, s1, s2, seed, bs=1024):
    model.eval(); ids = torch.as_tensor(idx_np, device=dev)
    gen = torch.Generator(device=dev); gen.manual_seed(1234 + 100 * seed)
    return torch.cat([torch.sigmoid(model(XG, ids[i:i + bs], s1, s2, gen)).float().cpu()
                      for i in range(0, len(ids), bs)]).numpy()

def train_nn(kind, cfg, seed, epochs, patience, want_test=False):
    torch.manual_seed(seed); np.random.seed(seed)
    if kind == "mlp":
        model = MLP(XG.shape[1], cfg["hidden"], cfg["layers"], cfg["drop"]).to(dev); s1 = s2 = None
    else:
        model = RelSAGE(XG.shape[1], cfg["hidden"], len(R_NAMES), cfg["drop"], cfg["hops_n"]).to(dev)
        s1, s2 = cfg["s1"], cfg["s2"]
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    posw = torch.tensor(float(spw * cfg["pw_mult"]), device=dev)
    tri = torch.as_tensor(tr_i, device=dev); bs = cfg["bs"]
    best, best_ep, bad, state = -1.0, 0, 0, None
    for ep in range(1, epochs + 1):
        model.train(); perm = tri[torch.randperm(len(tri), device=dev)]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = F.binary_cross_entropy_with_logits(model(XG, b, s1, s2), YT[b], pos_weight=posw)
            opt.zero_grad(); loss.backward(); opt.step()
        v = average_precision_score(yva_b, predict_avg(model, va_i, s1, s2, seed))
        if v > best:
            best, best_ep, bad = v, ep, 0
            state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    test = None
    if want_test:                                     # ТЕСТ рахується лише у фінальній комірці
        model.load_state_dict(state)
        test = metrics(yte_b, predict_avg(model, te_i, s1, s2, seed))
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return best, best_ep, test

def train_xgb(cfg, seed, want_test=False):
    m = XGBClassifier(n_estimators=2000, early_stopping_rounds=50, eval_metric="aucpr",
                      random_state=seed, **cfg, **DEV)
    m.fit(Xtr_b, ytr_b, eval_set=[(Xva_b, yva_b)], verbose=False)
    test = metrics(yte_b, m.predict_proba(Xte_b)[:, 1]) if want_test else None
    return float(m.best_score), int(m.best_iteration), test

# спроби повертають ЛИШЕ val (тест у пошуку не рахується)
xgb_trial = lambda cfg, seed: train_xgb(cfg, seed)[:2]
mlp_trial = lambda cfg, seed: train_nn("mlp", cfg, seed, 30, 5)[:2]
gnn_trial = lambda cfg, seed: train_nn("gnn", cfg, seed, 10, 3)[:2]

def lu(rng, a, b): return float(np.exp(rng.uniform(np.log(a), np.log(b))))
SPW_OPTS = [1.0, float(np.sqrt(spw)), spw]
def sample_xgb(rng):
    return dict(max_depth=int(rng.integers(4, 11)), learning_rate=lu(rng, 0.02, 0.3),
                subsample=float(rng.uniform(0.6, 1.0)), colsample_bytree=float(rng.uniform(0.3, 1.0)),
                min_child_weight=lu(rng, 1, 30), reg_lambda=lu(rng, 0.1, 30), reg_alpha=lu(rng, 1e-3, 5),
                gamma=float(rng.choice([0, 0.1, 0.5, 1, 2])), scale_pos_weight=float(rng.choice(SPW_OPTS)))
def sample_mlp(rng):
    return dict(hidden=int(rng.choice([64, 128, 256, 512])), layers=int(rng.choice([2, 3, 4])),
                drop=float(rng.choice([0.1, 0.2, 0.3, 0.4, 0.5])), lr=lu(rng, 3e-4, 3e-3), wd=lu(rng, 1e-6, 1e-3),
                bs=int(rng.choice([512, 1024, 2048])), pw_mult=float(rng.choice([0.25, 0.5, 1.0, 2.0])))
def sample_gnn(rng):
    return dict(hidden=int(rng.choice([64, 128, 256])), hops_n=int(rng.choice([1, 2])),
                s1=int(rng.choice([5, 10, 15])), s2=int(rng.choice([3, 5, 8])),
                drop=float(rng.choice([0.1, 0.2, 0.3, 0.4])), lr=lu(rng, 3e-4, 3e-3), wd=lu(rng, 1e-6, 1e-3),
                bs=int(rng.choice([512, 1024])), pw_mult=float(rng.choice([0.5, 1.0, 2.0])))

XGB_DEF = dict(max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, min_child_weight=1.0,
               reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, scale_pos_weight=spw)
MLP_DEF = dict(hidden=128, layers=2, drop=0.2, lr=1e-3, wd=1e-5, bs=1024, pw_mult=1.0)
GNN_DEF = dict(hidden=128, hops_n=2, s1=10, s2=5, drop=0.2, lr=1e-3, wd=1e-5, bs=1024, pw_mult=1.0)

JD = lambda o: o.item() if hasattr(o, "item") else str(o)
def load_log(name):
    p = f"results/ieee_cis_tuning_{name}.jsonl"
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

def run_tuning(name, sample_fn, trial_fn, default_cfg):
    path = f"results/ieee_cis_tuning_{name}.jsonl"
    log = load_log(name); used = sum(t["secs"] for t in log); budget = BUDGET_MIN * 60
    print(f"[{name}] вже спроб: {len(log)}, витрачено {used/60:.1f} хв із {BUDGET_MIN}")
    while used < budget:
        rng = np.random.default_rng(10_000 + len(log))
        cfg = default_cfg if len(log) == 0 else sample_fn(rng)       # спроба 0 = стандартна конфігурація
        t0 = time.time()
        try:
            val, ep = trial_fn(cfg, 0)
        except Exception as e:
            print("  збій спроби:", type(e).__name__, str(e)[:80]); val, ep = float("nan"), -1
            gc.collect(); torch.cuda.empty_cache()
        secs = time.time() - t0; used += secs
        rec = dict(trial=len(log), cfg=cfg, val=float(val), best_epoch=int(ep), secs=secs)
        log.append(rec)
        with open(path, "a") as f: f.write(json.dumps(rec, default=JD) + "\n")
        best = max([t["val"] for t in log if np.isfinite(t["val"])])
        print(f"  [{name}] спроба {rec['trial']}: val {val:.4f} (найкраща {best:.4f}) | {secs:.0f} с | всього {used/60:.1f} хв", flush=True)
    print(f"[{name}] ГОТОВО: {len(log)} спроб")
print("тренування й простори пошуку описано")

тренування й простори пошуку описано


In [5]:
# комірка 5
run_tuning("xgb", sample_xgb, xgb_trial, XGB_DEF)

[xgb] вже спроб: 0, витрачено 0.0 хв із 40
  [xgb] спроба 0: val 0.6690 (найкраща 0.6690) | 20 с | всього 0.3 хв
  [xgb] спроба 1: val 0.6637 (найкраща 0.6690) | 22 с | всього 0.7 хв
  [xgb] спроба 2: val 0.6958 (найкраща 0.6958) | 57 с | всього 1.7 хв
  [xgb] спроба 3: val 0.6823 (найкраща 0.6958) | 23 с | всього 2.0 хв
  [xgb] спроба 4: val 0.6715 (найкраща 0.6958) | 30 с | всього 2.5 хв
  [xgb] спроба 5: val 0.6655 (найкраща 0.6958) | 26 с | всього 3.0 хв
  [xgb] спроба 6: val 0.6808 (найкраща 0.6958) | 18 с | всього 3.3 хв
  [xgb] спроба 7: val 0.6708 (найкраща 0.6958) | 60 с | всього 4.3 хв
  [xgb] спроба 8: val 0.6872 (найкраща 0.6958) | 34 с | всього 4.8 хв
  [xgb] спроба 9: val 0.6814 (найкраща 0.6958) | 22 с | всього 5.2 хв
  [xgb] спроба 10: val 0.6912 (найкраща 0.6958) | 65 с | всього 6.3 хв
  [xgb] спроба 11: val 0.6559 (найкраща 0.6958) | 15 с | всього 6.5 хв
  [xgb] спроба 12: val 0.6749 (найкраща 0.6958) | 39 с | всього 7.2 хв
  [xgb] спроба 13: val 0.6832 (найкраща 0.69

In [6]:
# комірка 6
run_tuning("mlp", sample_mlp, mlp_trial, MLP_DEF)

[mlp] вже спроб: 0, витрачено 0.0 хв із 40
  [mlp] спроба 0: val 0.6281 (найкраща 0.6281) | 16 с | всього 0.3 хв
  [mlp] спроба 1: val 0.6294 (найкраща 0.6294) | 13 с | всього 0.5 хв
  [mlp] спроба 2: val 0.6232 (найкраща 0.6294) | 15 с | всього 0.8 хв
  [mlp] спроба 3: val 0.6216 (найкраща 0.6294) | 9 с | всього 0.9 хв
  [mlp] спроба 4: val 0.6318 (найкраща 0.6318) | 33 с | всього 1.5 хв
  [mlp] спроба 5: val 0.6143 (найкраща 0.6318) | 32 с | всього 2.0 хв
  [mlp] спроба 6: val 0.6359 (найкраща 0.6359) | 12 с | всього 2.2 хв
  [mlp] спроба 7: val 0.6255 (найкраща 0.6359) | 45 с | всього 2.9 хв
  [mlp] спроба 8: val 0.6194 (найкраща 0.6359) | 15 с | всього 3.2 хв
  [mlp] спроба 9: val 0.6228 (найкраща 0.6359) | 14 с | всього 3.4 хв
  [mlp] спроба 10: val 0.6200 (найкраща 0.6359) | 22 с | всього 3.8 хв
  [mlp] спроба 11: val 0.6306 (найкраща 0.6359) | 8 с | всього 3.9 хв
  [mlp] спроба 12: val 0.6240 (найкраща 0.6359) | 29 с | всього 4.4 хв
  [mlp] спроба 13: val 0.6308 (найкраща 0.6359

In [7]:
# комірка 7
run_tuning("gnn", sample_gnn, gnn_trial, GNN_DEF)

[gnn] вже спроб: 0, витрачено 0.0 хв із 40
  [gnn] спроба 0: val 0.6297 (найкраща 0.6297) | 315 с | всього 5.2 хв
  [gnn] спроба 1: val 0.6223 (найкраща 0.6297) | 166 с | всього 8.0 хв
  [gnn] спроба 2: val 0.6327 (найкраща 0.6327) | 203 с | всього 11.4 хв
  [gnn] спроба 3: val 0.6264 (найкраща 0.6327) | 311 с | всього 16.6 хв
  [gnn] спроба 4: val 0.6276 (найкраща 0.6327) | 92 с | всього 18.1 хв
  [gnn] спроба 5: val 0.6229 (найкраща 0.6327) | 21 с | всього 18.5 хв
  [gnn] спроба 6: val 0.6158 (найкраща 0.6327) | 26 с | всього 18.9 хв
  [gnn] спроба 7: val 0.6138 (найкраща 0.6327) | 140 с | всього 21.2 хв
  [gnn] спроба 8: val 0.6050 (найкраща 0.6327) | 127 с | всього 23.3 хв
  [gnn] спроба 9: val 0.6208 (найкраща 0.6327) | 522 с | всього 32.0 хв
  [gnn] спроба 10: val 0.5898 (найкраща 0.6327) | 8 с | всього 32.2 хв
  [gnn] спроба 11: val 0.6192 (найкраща 0.6327) | 34 с | всього 32.8 хв
  [gnn] спроба 12: val 0.6275 (найкраща 0.6327) | 17 с | всього 33.0 хв
  [gnn] спроба 13: val 0.60

Вибір, фінальні запуски, таблиця

In [8]:
def select(name, trial_fn, k, extra):
    L = [t for t in load_log(name) if np.isfinite(t["val"])]
    top = sorted(L, key=lambda t: -t["val"])[:k]; rows = []
    for t in top:
        vals = [t["val"]] + [trial_fn(t["cfg"], s)[0] for s in extra]
        rows.append(dict(trial=t["trial"], val_seed0=t["val"], val_mean=float(np.mean(vals)), cfg=t["cfg"]))
    return max(rows, key=lambda r: r["val_mean"]), len(load_log(name))

SEL = {}
for name, tf, k, extra in [("xgb", xgb_trial, 3, [1, 2]), ("mlp", mlp_trial, 3, [1, 2]), ("gnn", gnn_trial, 2, [1])]:
    best, n_tr = select(name, tf, k, extra)
    SEL[name] = dict(cfg=best["cfg"], trial=best["trial"], n_trials=n_tr, val_sel=best["val_mean"],
                     minutes=sum(t["secs"] for t in load_log(name)) / 60)
    print(f"{name}: обрано спробу {best['trial']} із {n_tr}, val (середнє за seed) = {best['val_mean']:.4f}")
    print("  ", best["cfg"], flush=True)
json.dump(SEL, open("results/ieee_cis_tuned_configs.json", "w"), indent=1, default=JD)

def final_runs(name, cfg):
    rows = []
    for sd in range(5):
        if name == "xgb": v, ep, te = train_xgb(cfg, sd, True)
        elif name == "mlp": v, ep, te = train_nn("mlp", cfg, sd, 30, 5, True)
        else: v, ep, te = train_nn("gnn", cfg, sd, 10, 3, True)
        rows.append(dict(model=name, seed=sd, val=v, best_epoch=ep, **te))
        print(f"  {name} seed {sd}: test PR-AUC {te['pr_auc']:.4f}", flush=True)
    return pd.DataFrame(rows)

FR = {}
for key, name, cfg in [("xgb tuned", "xgb", SEL["xgb"]["cfg"]), ("xgb default", "xgb", XGB_DEF),
                       ("mlp tuned", "mlp", SEL["mlp"]["cfg"]), ("mlp default", "mlp", MLP_DEF),
                       ("gnn tuned", "gnn", SEL["gnn"]["cfg"])]:
    print(key); FR[key] = final_runs(name, cfg)
pd.concat([d.assign(variant=k) for k, d in FR.items()]).to_csv("results/ieee_cis_tuning_final_5seeds.csv", index=False)

rows = []
for key, d in FR.items():
    nm = key.split()[0]; tuned = key.endswith("tuned")
    rows.append({"модель": key, "спроб": SEL[nm]["n_trials"] if tuned else "–",
                 "хв пошуку": round(SEL[nm]["minutes"]) if tuned else "–",
                 "val (5 seed)": d["val"].mean().round(4),
                 "test PR-AUC": f"{d.pr_auc.mean():.3f} ± {d.pr_auc.std():.3f}",
                 "ROC-AUC": d.roc_auc.mean().round(3), "Recall@k": d["recall@k"].mean().round(3),
                 "Precision@1%": d["precision@1%"].mean().round(3)})
summary = pd.DataFrame(rows); display(summary); summary.to_csv("results/ieee_cis_tuning_summary.csv", index=False)
print("довідково, ноутбук 13 (стандартні конфігурації, 15 епох у GNN): XGBoost 0.631 ± 0.001 | MLP 0.592 ± 0.003 | GNN 0.589 ± 0.006")

def paired(a, b):
    d = FR[a]["pr_auc"].values - FR[b]["pr_auc"].values; t, p = stats.ttest_1samp(d, 0.0)
    return d.mean(), d.std(ddof=1) / np.sqrt(len(d)), p
print("\nпарні різниці (test PR-AUC, 5 seed):")
for a, b, rule in [("xgb tuned", "xgb default", "тюнінг допоміг, якщо ≥ +0.01"),
                   ("mlp tuned", "mlp default", "тюнінг допоміг, якщо ≥ +0.01"),
                   ("gnn tuned", "mlp tuned", "перевага GNN, якщо > +0.01 і p < 0.025")]:
    m_, se_, p_ = paired(a, b); print(f"  {a} − {b}: {m_:+.4f} ± {se_:.4f} (SE), p = {p_:.3f}   [{rule}]")
mx = np.maximum(FR["gnn tuned"]["pr_auc"].values, FR["mlp tuned"]["pr_auc"].values)
d = FR["xgb tuned"]["pr_auc"].values - mx
print(f"  xgb tuned − max(gnn tuned, mlp tuned): {d.mean():+.4f} ± {d.std(ddof=1)/np.sqrt(5):.4f}   [перевага XGBoost, якщо > +0.01]")
print(f"  gnn tuned − 0.589 (референс ноутбука 13): {FR['gnn tuned'].pr_auc.mean() - 0.589:+.4f}")

xgb: обрано спробу 55 із 69, val (середнє за seed) = 0.6988
   {'max_depth': 9, 'learning_rate': 0.03336774698485237, 'subsample': 0.694885237588636, 'colsample_bytree': 0.3442855905516328, 'min_child_weight': 2.4238211336982842, 'reg_lambda': 0.5773776134099894, 'reg_alpha': 0.23594983181011883, 'gamma': 0.1, 'scale_pos_weight': 1.0}
mlp: обрано спробу 100 із 134, val (середнє за seed) = 0.6333
   {'hidden': 256, 'layers': 2, 'drop': 0.5, 'lr': 0.0003606648523061342, 'wd': 6.505333956635674e-06, 'bs': 512, 'pw_mult': 1.0}
gnn: обрано спробу 2 із 18, val (середнє за seed) = 0.6283
   {'hidden': 256, 'hops_n': 2, 's1': 5, 's2': 3, 'drop': 0.2, 'lr': 0.000305661246192521, 'wd': 8.691995730267805e-05, 'bs': 1024, 'pw_mult': 1.0}
xgb tuned


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [20:56:35] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  xgb seed 0: test PR-AUC 0.6530
  xgb seed 1: test PR-AUC 0.6497
  xgb seed 2: test PR-AUC 0.6557
  xgb seed 3: test PR-AUC 0.6491
  xgb seed 4: test PR-AUC 0.6512
xgb default
  xgb seed 0: test PR-AUC 0.6324
  xgb seed 1: test PR-AUC 0.6302
  xgb seed 2: test PR-AUC 0.6294
  xgb seed 3: test PR-AUC 0.6307
  xgb seed 4: test PR-AUC 0.6326
mlp tuned
  mlp seed 0: test PR-AUC 0.5982
  mlp seed 1: test PR-AUC 0.5951
  mlp seed 2: test PR-AUC 0.5971
  mlp seed 3: test PR-AUC 0.5921
  mlp seed 4: test PR-AUC 0.5970
mlp default
  mlp seed 0: test PR-AUC 0.5948
  mlp seed 1: test PR-AUC 0.5951
  mlp seed 2: test PR-AUC 0.5894
  mlp seed 3: test PR-AUC 0.5930
  mlp seed 4: test PR-AUC 0.5891
gnn tuned
  gnn seed 0: test PR-AUC 0.5935
  gnn seed 1: test PR-AUC 0.5962
  gnn seed 2: test PR-AUC 0.5974
  gnn seed 3: test PR-AUC 0.5917
  gnn seed 4: test PR-AUC 0.5913


,модель,спроб,хв пошуку,val (5 seed),test PR-AUC,ROC-AUC,Recall@k,Precision@1%
0,xgb tuned,69,40,0.6980,0.652 ± 0.003,0.943,0.596,0.940
1,xgb default,–,–,0.6687,0.631 ± 0.001,0.936,0.586,0.930
2,mlp tuned,134,40,0.6339,0.596 ± 0.002,0.925,0.566,0.895
3,mlp default,–,–,0.6275,0.592 ± 0.003,0.924,0.560,0.896
4,gnn tuned,18,43,0.6246,0.594 ± 0.003,0.927,0.564,0.903


довідково, ноутбук 13 (стандартні конфігурації, 15 епох у GNN): XGBoost 0.631 ± 0.001 | MLP 0.592 ± 0.003 | GNN 0.589 ± 0.006

парні різниці (test PR-AUC, 5 seed):
  xgb tuned − xgb default: +0.0207 ± 0.0015 (SE), p = 0.000   [тюнінг допоміг, якщо ≥ +0.01]
  mlp tuned − mlp default: +0.0037 ± 0.0019 (SE), p = 0.119   [тюнінг допоміг, якщо ≥ +0.01]
  gnn tuned − mlp tuned: -0.0019 ± 0.0014 (SE), p = 0.239   [перевага GNN, якщо > +0.01 і p < 0.025]
  xgb tuned − max(gnn tuned, mlp tuned): +0.0555 ± 0.0009   [перевага XGBoost, якщо > +0.01]
  gnn tuned − 0.589 (референс ноутбука 13): +0.0050
